Part 1:
Exercise 1:

In [20]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.spatial.distance import pdist
from scipy.stats import pearsonr, spearmanr
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
import glob
from pathlib import Path

from core import (
    MODEL_CLASSES,
    SantoroDataset,
    extract_activations,
    load_yamnet_activations,
)

In [21]:
# load dataset and make RDM
dataset = SantoroDataset()
brain_data = dataset.brain_responses

dl = DataLoader(dataset, batch_size=32, shuffle=False)

brain_rdm_vec = pdist(brain_data, metric="correlation")

def compute_rdm(data):
    flat = data.reshape(data.shape[0], -1)
    return pdist(flat, metric="correlation")


In [22]:
#  load model activations and make RDMs
activations = {}
results = []
model_names = ["waveform", "uninspired", "inspired"]
directories = [Path("models"), Path("assignment/models")]
for model_name in model_names:
    print(f"\nEvaluating architecture: {model_name}")

    # untrained models
    untrained_model = MODEL_CLASSES[model_name](num_classes=50)
    untrained_model.eval()
    untrained_acts = extract_activations(dl, untrained_model)
    activations[f"{model_name}_Untrained"] = untrained_acts

    for layer_idx, (layer_name, acts) in enumerate(untrained_acts.items()):
        m_vec = compute_rdm(acts)
        rho, p_val = spearmanr(m_vec, brain_rdm_vec)
        results.append(
            {
                "Model": model_name,
                "Condition": "Untrained",
                "Run": "untrained",
                "Layer_Idx": layer_idx,
                "Layer_Name": layer_name,
                "Spearman_r": rho,
                "p_val": p_val,
            }
        )

    # trained models
    ckpt_files = []
    for d in directories:
        ckpt_files.extend(sorted(glob.glob(str(d / f"{model_name}*run*.pt"))))

    if not ckpt_files:
        print(f"  Warning: No .pt files found for {model_name} in {directories}. Skipping trained model evaluation.")
        continue

    print(f"  Found {len(ckpt_files)} checkpoints for {model_name}")

    for ckpt_path in ckpt_files:
        run_name = Path(ckpt_path).stem.split("_")[-2]  # e.g., 'run0'
        model = MODEL_CLASSES[model_name](num_classes=50)

        checkpoint = torch.load(
            ckpt_path, map_location="cpu", weights_only=False
        )
        if isinstance(checkpoint, dict):
            if "model_state_dict" in checkpoint:
                state_dict = checkpoint["model_state_dict"]
            elif "state_dict" in checkpoint:
                state_dict = checkpoint["state_dict"]
            else:
                state_dict = checkpoint
        else:
            state_dict = checkpoint

        model.load_state_dict(state_dict)
        model.eval()

        layer_acts = extract_activations(dl, model)
        # Store run0 activations for t-SNE visualization
        if run_name == "run0":
            activations[f"{model_name}_Trained"] = layer_acts

        for layer_idx, (layer_name, acts) in enumerate(layer_acts.items()):
            m_vec = compute_rdm(acts)
            rho, p_val = spearmanr(m_vec, brain_rdm_vec)
            results.append(
                {
                    "Model": model_name,
                    "Condition": "Trained",
                    "Run": run_name,
                    "Layer_Idx": layer_idx,
                    "Layer_Name": layer_name,
                    "Spearman_r": rho,
                    "p_val": p_val,
                }
            )


Evaluating architecture: waveform
  Found 5 checkpoints for waveform

Evaluating architecture: uninspired
  Found 5 checkpoints for uninspired

Evaluating architecture: inspired
  Found 5 checkpoints for inspired


In [23]:
# load yamnet activations and make RDM
yamnet_activations = load_yamnet_activations()
activations["yamnet"] = yamnet_activations

for layer_idx, (layer_name, acts) in enumerate(yamnet_activations.items()):
    m_vec = compute_rdm(acts)
    rho, p_val = spearmanr(m_vec, brain_rdm_vec)
    results.append(
        {
            "Model": "yamnet",
            "Condition": "Pretrained",
            "Run": "pretrained",
            "Layer_Idx": layer_idx,
            "Layer_Name": layer_name,
            "Spearman_r": rho,
            "p_val": p_val,
        }
    )

df_rsa = pd.DataFrame(results)
print("\nRSA Summary:")
print(
    df_rsa.groupby(["Model", "Condition"])["Spearman_r"].max().reset_index()
)


RSA Summary:
        Model   Condition  Spearman_r
0    inspired     Trained    0.067118
1    inspired   Untrained    0.074233
2  uninspired     Trained    0.037479
3  uninspired   Untrained    0.068145
4    waveform     Trained    0.006758
5    waveform   Untrained   -0.019805
6      yamnet  Pretrained    0.098000


I loaded the santoro dataset and computed the RDM for the brain responses. Then, I loaded the activations from different models (both untrained and trained) and computed their RDMs for each layer (5 layers each). I also loaded the activations from the pretrained yamnet model and computed its RDM across layers. Finally, I compared the model RDMs with the brain RDM using Spearman correlation and summarized the results in a DataFrame. The spearman correlation values indicate how well each model's representations align with the brain's representations for the same stimuli.